In [ ]:
# =====================================
# Analyse univariée – Données milieu marin
# =====================================

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis

# -------------------------------
# 0. Charger les données
# -------------------------------
url = "https://github.com/simaleo/data_book1/raw/main/milieu_marin_700.csv"
data = pd.read_csv(url)

# Définir les variables qualitatives et quantitatives
qual_vars = ['Station', 'Type_milieu', 'Algues']
quant_vars = ['Température_C', 'Salinité_PSU', 'pH', 'Oxygène_dissous_mgL', 'Nitrates_uM', 'Phytoplancton_cells_mL']

# -------------------------------
# I. Introduction
# -------------------------------
print("=== INTRODUCTION ===")
print(f"Nombre total d'observations : {data.shape[0]}")
print(f"Nombre de variables : {data.shape[1]}\n")

# -------------------------------
# II. Variables qualitatives
# -------------------------------
print("=== VARIABLES QUALITATIVES ===")
for var in qual_vars:
    print(f"\nVariable : {var}")
    counts = data[var].value_counts()
    percents = data[var].value_counts(normalize=True) * 100
    print("Fréquences absolues :")
    print(counts)
    print("Fréquences relatives (%) :")
    print(percents)

    # Diagramme en barres
    plt.figure(figsize=(8,5))
    sns.countplot(x=var, data=data, palette='viridis')
    plt.title(f"Distribution de la variable {var}")
    plt.ylabel("Effectif")
    plt.xlabel(var)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# -------------------------------
# III. Variables quantitatives
# -------------------------------
print("=== VARIABLES QUANTITATIVES ===")
quant_summary = pd.DataFrame(columns=['Variable','Moyenne','Médiane','Min','Max','Étendue',
                                      'Variance','Écart-type','CV (%)','Q1','Q3','IQR',
                                      'Skewness','Kurtosis','Nb_outliers'])

for var in quant_vars:
    # Paramètres de position et dispersion
    moyenne = data[var].mean()
    mediane = data[var].median()
    min_val = data[var].min()
    max_val = data[var].max()
    etendue = max_val - min_val
    variance = data[var].var()
    ecart_type = data[var].std()
    cv = (ecart_type / moyenne) * 100
    Q1 = data[var].quantile(0.25)
    Q3 = data[var].quantile(0.75)
    IQR = Q3 - Q1
    skew_val = skew(data[var])
    kurt_val = kurtosis(data[var])
    outliers = data[(data[var] < Q1 - 1.5*IQR) | (data[var] > Q3 + 1.5*IQR)]
    nb_outliers = len(outliers)

    # Ajouter au tableau récapitulatif
    quant_summary = pd.concat([quant_summary, pd.DataFrame([{
        'Variable':var,'Moyenne':moyenne,'Médiane':mediane,'Min':min_val,'Max':max_val,
        'Étendue':etendue,'Variance':variance,'Écart-type':ecart_type,'CV (%)':cv,
        'Q1':Q1,'Q3':Q3,'IQR':IQR,'Skewness':skew_val,'Kurtosis':kurt_val,
        'Nb_outliers':nb_outliers
    }])], ignore_index=True)

    # Histogramme et boxplot
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    sns.histplot(data[var], kde=True, bins=20, color='skyblue')
    plt.title(f"Histogramme de {var}")
    plt.subplot(1,2,2)
    sns.boxplot(y=data[var], color='lightgreen')
    plt.title(f"Boxplot de {var}")
    plt.tight_layout()
    plt.show()

# -------------------------------
# IV. Tableau récapitulatif
# -------------------------------
print("\n=== TABLEAU RÉCAPITULATIF DES VARIABLES QUANTITATIVES ===")
print(quant_summary)

# -------------------------------
# V. Visualisation globale
# -------------------------------
# Boxplots groupés par Type_milieu
for var in quant_vars:
    plt.figure(figsize=(8,5))
    sns.boxplot(x='Type_milieu', y=var, data=data, palette='Set2')
    plt.title(f"{var} selon le Type de milieu")
    plt.show()

# Boxplots groupés par Algues
for var in quant_vars:
    plt.figure(figsize=(8,5))
    sns.boxplot(x='Algues', y=var, data=data, palette='Set3')
    plt.title(f"{var} selon la présence d'algues")
    plt.show()

# Histogrammes multiples pour comparaison
data[quant_vars].hist(bins=20, figsize=(15,10), color='lightblue', edgecolor='black')
plt.suptitle("Histogrammes des variables quantitatives")
plt.show()

# -------------------------------
# VI. Standardisation (optionnelle)
# -------------------------------
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
data_scaled = pd.DataFrame(scaler.fit_transform(data[quant_vars]), columns=quant_vars)
print("\n=== STANDARDISATION DES VARIABLES QUANTITATIVES ===")
print(data_scaled.head())